# LightGBM con una CRNN pequeña para puntuar secuencias de sueño

**Resultado de cinco rondas por persona.** Cada uno de los 75.612 segmentos de las 56 noches recibió predicciones de modelos entrenados sin esa persona. Con una mezcla fija de 75 % DART y 25 % CRNN, el F1 macro pasó de **80,10 % a 80,86 %**. La combinación mejoró en 5 de las cinco rondas y en 26 de las 34 personas.

| Modelo | F1 macro global | Exactitud |
|---|---:|---:|
| DART | 80,10 % | 85,96 % |
| DART temporal | 79,67 % | 85,16 % |
| CRNN pequeña | 79,56 % | 85,90 % |
| DART + CRNN | 80,86 % | 86,69 % |

Primero se exploraron el ajuste por transiciones y la CRNN sobre la prueba original y la primera ronda. Luego se mantuvo la configuración para la validación de cinco rondas. Los mismos datos ya habían intervenido en el desarrollo del proyecto. Una cohorte externa permitiría estimar cuánto de la mejora se mantiene en personas recogidas en otras condiciones.

La decodificación por transiciones se inspira en trabajos que combinan un clasificador con una secuencia de etapas. [Large-Scale Automated Sleep Staging](https://pmc.ncbi.nlm.nih.gov/articles/PMC6251659/).

In [1]:
import json
import sys
from pathlib import Path
from tempfile import gettempdir
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import Booster
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data.dvc').is_file())
sys.path.insert(0, str(ROOT / 'packages/sleep-staging/src'))
sys.path.insert(0, str(ROOT))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_telemetry_records, discover_sleep_cassette_records
from sleep_staging.training import load_or_build_supervised_dataset
from scripts.compare_cassette_hybrid import decode_night

ref = ROOT / 'data/experimentos/lightgbm_st_sc'
spec = json.loads((ref / 'experimento.json').read_text('utf-8'))
clases = np.array(spec['classes'])
fuerza = 0.35
def datos(registros, cohorte, cache):
    cfg = spec['preprocessing_telemetry' if cohorte == 'Telemetry' else 'preprocessing_cassette']
    ds = load_or_build_supervised_dataset(registros, PreprocessingPipeline(PreprocessingConfig(**cfg)), cache, verbose=False)
    x = ds.features.astype('float32').reindex(columns=spec['feature_columns'])
    m = ds.metadata[['record_id','subject_id','epoch_index']].copy()
    m['cohorte'], m['real'] = cohorte, ds.labels.to_numpy()
    return x, m
st = discover_sleep_telemetry_records(ROOT/'data/sleep-telemetry')
sc = discover_sleep_cassette_records(ROOT/'data/sleep-cassette')
partes = [datos(st,'Telemetry',ROOT/'data/processed/supervised')]
partes.extend(datos([r],'Cassette',Path(gettempdir())/'sleep-edfx-sc-features-v1') for r in sc)
X = pd.concat([x for x,_ in partes], ignore_index=True)
meta = pd.concat([m for _,m in partes], ignore_index=True)
del partes
assert not meta.duplicated(['record_id','epoch_index']).any() and set(meta.real)==set(clases)

C:\Users\sebas\AppData\Local\Temp\ipykernel_21076\1900523081.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  X = pd.concat([x for x,_ in partes], ignore_index=True)


## Decodificación por noche

Se estiman transiciones entre segmentos consecutivos de las personas de entrenamiento, con un pseudoconteo por celda. El factor temporal es la probabilidad de transición dividida por la frecuencia de destino, elevada a 0,35. Así se pondera la información que aporta la etapa anterior. Viterbi combina ese factor con las probabilidades del DART. Se inicia cada secuencia con pesos iguales y se reinicia ante un hueco o cambio de noche. Esta evaluación utiliza los segmentos con anotación disponible en la tabla supervisada.

In [2]:
def factores(entrenamiento):
    k = len(clases)
    cuenta = np.ones((k,k), dtype=float)
    for _, noche in entrenamiento.groupby('record_id'):
        noche = noche.sort_values('epoch_index')
        idx = pd.Categorical(noche.real, categories=clases).codes
        juntos = np.diff(noche.epoch_index)==1
        np.add.at(cuenta,(idx[:-1][juntos],idx[1:][juntos]),1)
    A = cuenta / cuenta.sum(axis=1, keepdims=True)
    destino = cuenta.sum(axis=0) / cuenta.sum()
    return (A / destino)**fuerza

def evaluar(archivo, personas):
    prueba = meta.subject_id.isin(personas)
    assert set(meta.loc[prueba,'subject_id']) == set(personas)
    trans = factores(meta.loc[~prueba])
    m = meta.loc[prueba].copy().reset_index(drop=True)
    proba = Booster(model_file=str(archivo)).predict(X.loc[prueba], num_threads=4)
    assert proba.shape==(len(m),len(clases)) and np.allclose(proba.sum(axis=1),1)
    m['DART'] = clases[proba.argmax(axis=1)]
    secuencia = np.full(len(m), -1, dtype=int)
    for _, noche in m.groupby('record_id'):
        noche = noche.sort_values('epoch_index')
        cortes = np.flatnonzero(np.diff(noche.epoch_index)!=1)+1
        for filas in np.split(noche.index.to_numpy(), cortes):
            secuencia[filas] = decode_night(proba[filas], trans, np.ones(len(clases)))
    assert (secuencia>=0).all()
    m['DART temporal'] = clases[secuencia]
    return m

assert set(spec['train_subjects']).isdisjoint(spec['validation_subjects'])
assert set(spec['train_subjects']) | set(spec['validation_subjects']) == set(meta.subject_id)
resultados = {'Prueba original': evaluar(ref/'model.txt',spec['validation_subjects'])}
local = ROOT/'.sleep-cv/8f83a22b1d3a65d3'
pesos = local/'tiempos-ronda-1/dart.txt'
if pesos.exists() and (local/'fold-1-dart.parquet').exists():
    guardado = pd.read_parquet(local/'fold-1-dart.parquet')
    assert json.loads((pesos.parent/'mediciones.json').read_text())['clases_dart']==clases.tolist()
    r = evaluar(pesos,guardado.subject.unique())
    par = r.merge(guardado,on=['record_id','epoch_index'], validate='one_to_one')
    assert len(par)==len(r)==len(guardado) and (par.real==par.stage).all() and (par.DART==par.prediction).all()
    resultados['Primera ronda'] = r

## Resultados sobre las mismas personas

Se calcula el F1 macro de las cinco etapas en los segmentos combinados. El intervalo de la diferencia se estima con 5.000 remuestreos pareados por persona, conservando el número de personas de cada cohorte. Con pocos sujetos el intervalo es orientativo.

In [3]:
def intervalo(m):
    rng = np.random.default_rng(42)
    matrices = []
    for _,g in m.groupby('cohorte'):
        cms = np.array([[confusion_matrix(s.real,s[n],labels=clases) for n in ['DART','DART temporal']]
                        for _,s in g.groupby('subject_id')])
        indices = rng.integers(len(cms),size=(5000,len(cms)))
        matrices.append(cms[indices].sum(axis=1))
    cm = sum(matrices)
    denom = cm.sum(axis=-1)+cm.sum(axis=-2)
    por_clase = np.divide(2*np.diagonal(cm,axis1=-2,axis2=-1),denom,
                          out=np.zeros(denom.shape),where=denom!=0)
    f1 = por_clase.mean(axis=-1)
    return np.quantile(f1[:,1]-f1[:,0],[0.025,0.975])

filas, etapas, diferencias = [], [], []
for corte,m in resultados.items():
    for cohorte,g in [('Ambas',m),*m.groupby('cohorte')]:
        for modelo in ['DART','DART temporal']:
            filas.append([corte,cohorte,modelo,g.subject_id.nunique(),g.record_id.nunique(),len(g),
                         f1_score(g.real,g[modelo],labels=clases,average='macro',zero_division=0),
                         accuracy_score(g.real,g[modelo]),cohen_kappa_score(g.real,g[modelo])])
    for modelo in ['DART','DART temporal']:
        etapas.append([corte,modelo,*f1_score(m.real,m[modelo],labels=clases,average=None,zero_division=0)])
    d = f1_score(m.real,m['DART temporal'],labels=clases,average='macro')-f1_score(m.real,m.DART,labels=clases,average='macro')
    diferencias.append([corte,d,*intervalo(m)])
display(pd.DataFrame(filas,columns=['Corte','Cohorte','Modelo','Personas','Noches','Segmentos','F1 macro','Exactitud','Kappa']).round(4))
display(pd.DataFrame(etapas,columns=['Corte','Modelo',*clases]).round(4))
display(pd.DataFrame(diferencias,columns=['Corte','Diferencia F1','IC 95 % inferior','IC 95 % superior']).round(4))

,Corte,Cohorte,Modelo,Personas,Noches,Segmentos,F1 macro,Exactitud,Kappa
0,Prueba original,Ambas,DART,8,13,18378,0.7981,0.8504,0.7942
1,Prueba original,Ambas,DART temporal,8,13,18378,0.7986,0.8466,0.7906
2,Prueba original,Cassette,DART,3,3,8449,0.7686,0.8884,0.7608
3,Prueba original,Cassette,DART temporal,3,3,8449,0.7735,0.8882,0.7634
4,Prueba original,Telemetry,DART,5,10,9929,0.7948,0.8181,0.7541
5,Prueba original,Telemetry,DART temporal,5,10,9929,0.7885,0.8113,0.7461
6,Primera ronda,Ambas,DART,7,11,15893,0.8194,0.8745,0.8262
7,Primera ronda,Ambas,DART temporal,7,11,15893,0.8075,0.8618,0.8106
8,Primera ronda,Cassette,DART,3,3,8039,0.7394,0.9167,0.8296
9,Primera ronda,Cassette,DART temporal,3,3,8039,0.7245,0.9070,0.8127


,Corte,Modelo,N1,N2,N3,REM,W
0,Prueba original,DART,0.5406,0.8124,0.8237,0.8788,0.9350
1,Prueba original,DART temporal,0.5482,0.8047,0.8237,0.8819,0.9342
2,Primera ronda,DART,0.6373,0.8320,0.7887,0.8631,0.9757
3,Primera ronda,DART temporal,0.6158,0.8097,0.7731,0.8641,0.9750


,Corte,Diferencia F1,IC 95 % inferior,IC 95 % superior
0,Prueba original,0.0004,-0.0061,0.0065
1,Primera ronda,-0.0118,-0.0186,-0.0061


## CRNN pequeña sobre características y combinación con DART

Como segunda hipótesis exploratoria, probamos una Conv1D y una GRU bidireccional sobre las características del segmento actual de 30 s. La red aprende a puntuar secuencias completas. Usa ventanas de 64 segmentos con 32 de desplazamiento, sin cruzar noches ni saltos de tiempo. La mezcla se fija en 75 % de la probabilidad de DART y 25 % de la de la red. Se entrenan redes nuevas en cada corte durante 12 épocas sin seleccionar épocas ni pesos de mezcla con los sujetos de prueba. La comparación es para puntuar noches completas fuera de línea. [SeqSleepNet](https://arxiv.org/abs/1809.10932)

In [4]:
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from time import perf_counter

try:
    tf.config.set_visible_devices([], 'GPU')
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # Un kernel que ya inicializó TensorFlow conserva su configuración.
instantaneas = [c for c in X if c.endswith('_t0')]
base_neural = X[instantaneas].to_numpy(dtype='float32')
codigos = pd.Categorical(meta.real, categories=clases).codes.astype('int32')

def ventanas(datos):
    indices = []
    for _, noche in datos.groupby('record_id'):
        noche = noche.sort_values('epoch_index')
        for tramo in np.split(noche.index.to_numpy(), np.flatnonzero(np.diff(noche.epoch_index)!=1)+1):
            for inicio in range(0, len(tramo), 32):
                v = tramo[inicio:inicio+64]
                indices.append(np.pad(v,(0,64-len(v)),constant_values=-1))
                if inicio+64 >= len(tramo):
                    break
    return np.stack(indices)

def red_secuencial(dim):
    entrada = tf.keras.Input((64,dim))
    # La última variable indica qué posiciones contienen un segmento real.
    mascara = tf.keras.ops.greater(entrada[:,:,-1],0)
    z = tf.keras.layers.Conv1D(48,3,padding='same',activation='relu')(entrada)
    z = tf.keras.layers.Dropout(0.15)(z)
    z = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(32,return_sequences=True))(z, mask=mascara)
    z = tf.keras.layers.Dropout(0.2)(z)
    salida = tf.keras.layers.Dense(len(clases),activation='softmax')(z)
    red = tf.keras.Model(entrada,salida)
    red.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy')
    return red

def probar_red(corte, pesos, semilla):
    m = resultados[corte].copy()
    es_test = meta.subject_id.isin(m.subject_id.unique())
    tr, te = meta.loc[~es_test], meta.loc[es_test]
    assert set(tr.subject_id).isdisjoint(te.subject_id)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(semilla)
    escala = RobustScaler(quantile_range=(10,90)).fit(base_neural[~es_test])
    z = np.nan_to_num(np.clip(escala.transform(base_neural),-5,5)).astype('float32')
    falta_emg = meta.cohorte.eq('Cassette').to_numpy()
    z = np.column_stack([z, falta_emg.astype('float32'), np.ones(len(z),dtype='float32')])
    entradas = np.vstack([z,np.zeros((1,z.shape[1]),dtype='float32')])
    # El índice -1 apunta a la fila de relleno, cuya pérdida se pondera a cero.
    a,b = ventanas(tr),ventanas(te)
    assert set(a[a>=0])==set(tr.index) and set(b[b>=0])==set(te.index)
    counts = np.bincount(codigos[tr.index],minlength=len(clases))
    w = (counts.mean()/counts)**0.35
    w /= np.average(w, weights=counts)
    red = red_secuencial(z.shape[1])
    inicio = perf_counter()
    red.fit(entradas[a],codigos[np.maximum(a,0)],sample_weight=(w[codigos[np.maximum(a,0)]]*(a>=0)).astype('float32'),
            epochs=12,batch_size=32,verbose=0)
    print(corte, '· ajuste CRNN en segundos:', round(perf_counter()-inicio,1), flush=True)
    partes = red.predict(entradas[b],batch_size=64,verbose=0)
    suma = np.zeros((len(meta), len(clases)))
    votos = np.zeros(len(meta))
    validos = b>=0
    np.add.at(suma,b[validos],partes[validos])
    np.add.at(votos,b[validos],1)
    neural = suma[te.index]/votos[te.index,None]
    tabular = Booster(model_file=str(pesos)).predict(X.loc[te.index], num_threads=4)
    assert np.array_equal(te.record_id.to_numpy(),m.record_id) and np.array_equal(te.epoch_index.to_numpy(),m.epoch_index)
    assert np.allclose(neural.sum(1),1) and np.array_equal(clases[tabular.argmax(1)],m.DART)
    m['CRNN pequeña'] = clases[neural.argmax(1)]
    m['DART + CRNN'] = clases[(0.75*tabular+0.25*neural).argmax(1)]
    return m

fusiones = {'Prueba original': probar_red('Prueba original',ref/'model.txt',42)}
if 'Primera ronda' in resultados:
    fusiones['Primera ronda'] = probar_red('Primera ronda',pesos,42)

Prueba original · ajuste CRNN en segundos: 15.3


Primera ronda · ajuste CRNN en segundos: 16.2


In [5]:
tabla, incertidumbre = [], []
for corte,m in fusiones.items():
    for cohorte,g in [('Ambas',m),*m.groupby('cohorte')]:
        for modelo in ['DART','CRNN pequeña','DART + CRNN']:
            por_clase = f1_score(g.real,g[modelo],labels=clases,average=None,zero_division=0)
            tabla.append([corte,cohorte,modelo,por_clase.mean(),*por_clase,accuracy_score(g.real,g[modelo])])
    ic = intervalo(m.drop(columns='DART temporal').rename(columns={'DART + CRNN':'DART temporal'}))
    delta = f1_score(m.real,m['DART + CRNN'],average='macro')-f1_score(m.real,m.DART,average='macro')
    incertidumbre.append([corte,delta,*ic])
display(pd.DataFrame(tabla,columns=['Corte','Cohorte','Modelo','F1 macro',*clases,'Exactitud']).round(4))
display(pd.DataFrame(incertidumbre,columns=['Corte','Diferencia F1 de la mezcla','IC 95 % inferior','IC 95 % superior']).round(4))

,Corte,Cohorte,Modelo,F1 macro,N1,N2,N3,REM,W,Exactitud
0,Prueba original,Ambas,DART,0.7981,0.5406,0.8124,0.8237,0.8788,0.9350,0.8504
1,Prueba original,Ambas,CRNN pequeña,0.7900,0.4926,0.8163,0.8043,0.8952,0.9416,0.8500
2,Prueba original,Ambas,DART + CRNN,0.8046,0.5369,0.8240,0.8223,0.8952,0.9448,0.8595
3,Prueba original,Cassette,DART,0.7686,0.4595,0.7779,0.8371,0.8189,0.9496,0.8884
4,Prueba original,Cassette,CRNN pequeña,0.7736,0.4151,0.8006,0.7863,0.9057,0.9602,0.8961
5,Prueba original,Cassette,DART + CRNN,0.7901,0.4647,0.8075,0.8379,0.8788,0.9616,0.9048
6,Prueba original,Telemetry,DART,0.7948,0.5736,0.8248,0.8226,0.8945,0.8585,0.8181
7,Prueba original,Telemetry,CRNN pequeña,0.7796,0.5303,0.8224,0.8060,0.8922,0.8469,0.8108
8,Prueba original,Telemetry,DART + CRNN,0.7948,0.5661,0.8302,0.8210,0.8997,0.8570,0.8209
9,Primera ronda,Ambas,DART,0.8194,0.6373,0.8320,0.7887,0.8631,0.9757,0.8745


,Corte,Diferencia F1 de la mezcla,IC 95 % inferior,IC 95 % superior
0,Prueba original,0.0065,-0.0014,0.0154
1,Primera ronda,0.0080,0.0018,0.0139


## Comprobación en las cinco rondas por persona

Se conserva la fórmula 75/25 y la arquitectura ya probada. Se vuelve a entrenar cada clasificador únicamente con las personas de su ronda. Las cinco rondas usan las mismas asignaciones del notebook de comparación. Cada persona aporta una sola predicción de prueba por segmento al resultado conjunto. La elección exploratoria de esta combinación ya utilizó resultados de este dataset, por lo que sigue pendiente una prueba externa. Los pesos DART y las predicciones por ronda se guardan en la caché local para reanudar.

In [6]:
import hashlib
import os
from tempfile import NamedTemporaryFile
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

archivo_nb = ROOT/'Notebooks/experiments/lightgbm_transiciones_sueno.ipynb'
codigo = [c['source'] for c in json.loads(archivo_nb.read_text('utf-8'))['cells'] if c['cell_type']=='code']
firma = hashlib.sha256(json.dumps([codigo,lgb.__version__,tf.__version__]).encode())
firma.update((ROOT/'data.dvc').read_bytes())
for fuente in [ROOT/'scripts/compare_cassette_hybrid.py', *sorted((ROOT/'packages/sleep-staging/src/sleep_staging').rglob('*.py'))]:
    firma.update(fuente.read_bytes())
# La caché depende del código de lightgbm_transiciones_sueno, los datos y las versiones.
aliases = json.loads(archivo_nb.read_text('utf-8'))['metadata']['cache_tras_traslado']
cache_cv = ROOT/'.sleep-cv/fusion-crnn'/aliases.get(firma.hexdigest(), firma.hexdigest()[:16])
cache_cv.mkdir(parents=True,exist_ok=True)
personas = meta[['subject_id','cohorte']].drop_duplicates().sort_values(['cohorte','subject_id'])
assert personas.subject_id.is_unique
rondas = [set(personas.iloc[b].subject_id) for _,b in StratifiedKFold(5,shuffle=True,random_state=42).split(personas,personas.cohorte)]
assert len(set.union(*rondas)) == sum(map(len,rondas)) == meta.subject_id.nunique()

def comprobar_ronda(i):
    sujetos = rondas[i-1]
    etiqueta = f'Ronda {i} CV'
    ruta = cache_cv/f'ronda-{i}.parquet'
    if ruta.exists():
        m = pd.read_parquet(ruta)
    elif i == 1 and 'Primera ronda' in fusiones:
        m = fusiones['Primera ronda'].copy()
    else:
        modelo = cache_cv/f'dart-{i}.txt'
        if not modelo.exists():
            es_train = ~meta.subject_id.isin(sujetos)
            ajustado = lgb.LGBMClassifier(**dict(spec['parameters'],n_jobs=6))
            ajustado.fit(X.loc[es_train],meta.real.loc[es_train])
            assert np.array_equal(ajustado.classes_,clases)
            with NamedTemporaryFile(dir=cache_cv,delete=False) as tmp:
                temporal = Path(tmp.name)
            ajustado.booster_.save_model(str(temporal))
            os.replace(temporal,modelo)
            del ajustado
        resultados[etiqueta] = evaluar(modelo,sujetos)
        m = probar_red(etiqueta,modelo,42)
    cols = ['record_id','epoch_index','subject_id','real','cohorte']
    base = meta.loc[meta.subject_id.isin(sujetos),cols]
    pd.testing.assert_frame_equal(m[cols].sort_values(cols).reset_index(drop=True),base.sort_values(cols).reset_index(drop=True))
    assert m[['DART','DART temporal','CRNN pequeña','DART + CRNN']].isin(clases).all().all()
    previo = local/f'fold-{i}-dart.parquet'
    if previo.is_file():
        g = pd.read_parquet(previo).rename(columns={'subject':'subject_id','stage':'real'})
        par = m.merge(g,on=['record_id','epoch_index','subject_id','real'],validate='one_to_one')
        assert len(g)==len(par)==len(m) and np.array_equal(par.DART,par.prediction)
    if not ruta.exists():
        with NamedTemporaryFile(dir=cache_cv,delete=False) as tmp:
            temporal = Path(tmp.name)
        m.to_parquet(temporal,index=False)
        os.replace(temporal,ruta)
    print(etiqueta, '· personas',len(sujetos), '· segmentos',len(m), '· F1 macro')
    for modelo in ['DART','CRNN pequeña','DART + CRNN']:
        print(f"{modelo}: {f1_score(m.real,m[modelo],labels=clases,average='macro'):.4f}")

In [7]:
comprobar_ronda(1)

Ronda 1 CV · personas 7 · segmentos 15893 · F1 macro
DART: 0.8194
CRNN pequeña: 0.8052
DART + CRNN: 0.8274


In [8]:
comprobar_ronda(2)

Ronda 2 CV · ajuste CRNN en segundos: 16.2


Ronda 2 CV · personas 7 · segmentos 15763 · F1 macro
DART: 0.8025
CRNN pequeña: 0.8054
DART + CRNN: 0.8155


In [9]:
comprobar_ronda(3)

Ronda 3 CV · ajuste CRNN en segundos: 17.3


Ronda 3 CV · personas 7 · segmentos 15409 · F1 macro
DART: 0.8075
CRNN pequeña: 0.8076
DART + CRNN: 0.8181


In [10]:
comprobar_ronda(4)

Ronda 4 CV · ajuste CRNN en segundos: 16.7


Ronda 4 CV · personas 7 · segmentos 14989 · F1 macro
DART: 0.8071
CRNN pequeña: 0.8053
DART + CRNN: 0.8128


In [11]:
comprobar_ronda(5)

Ronda 5 CV · ajuste CRNN en segundos: 17.9


Ronda 5 CV · personas 6 · segmentos 13558 · F1 macro
DART: 0.7598
CRNN pequeña: 0.7402
DART + CRNN: 0.7599


In [12]:
oof = pd.concat([pd.read_parquet(cache_cv/f'ronda-{i}.parquet').assign(ronda=i) for i in range(1,6)],ignore_index=True)
assert len(oof)==len(meta) and oof.groupby('subject_id').ronda.nunique().eq(1).all()
assert not oof.duplicated(['record_id','epoch_index']).any()
resumen = []
for cohorte,g in [('Ambas',oof),*oof.groupby('cohorte')]:
    for modelo in ['DART','DART temporal','CRNN pequeña','DART + CRNN']:
        scores = f1_score(g.real,g[modelo],labels=clases,average=None,zero_division=0)
        resumen.append([cohorte,modelo,g.subject_id.nunique(),g.record_id.nunique(),len(g),
                        scores.mean(),accuracy_score(g.real,g[modelo]),*scores])
display(pd.DataFrame(resumen,columns=['Cohorte','Modelo','Personas','Noches','Segmentos','F1 macro','Exactitud',*clases]).round(4))
for modelo in ['DART temporal','DART + CRNN']:
    comparacion = oof.copy()
    comparacion['DART temporal'] = oof[modelo]
    ic = intervalo(comparacion)
    delta = f1_score(oof.real,oof[modelo],average='macro')-f1_score(oof.real,oof.DART,average='macro')
    print(f'{modelo} frente a DART · diferencia {delta:+.4f} · IC 95 % por sujeto [{ic[0]:+.4f}, {ic[1]:+.4f}]')

,Cohorte,Modelo,Personas,Noches,Segmentos,F1 macro,Exactitud,N1,N2,N3,REM,W
0,Ambas,DART,34,56,75612,0.8010,0.8596,0.5425,0.8384,0.7949,0.8770,0.9520
1,Ambas,DART temporal,34,56,75612,0.7967,0.8516,0.5357,0.8249,0.7878,0.8831,0.9519
2,Ambas,CRNN pequeña,34,56,75612,0.7956,0.8590,0.5319,0.8493,0.7739,0.8745,0.9484
3,Ambas,DART + CRNN,34,56,75612,0.8086,0.8669,0.5563,0.8489,0.7974,0.8846,0.9561
4,Cassette,DART,12,12,32921,0.7768,0.9144,0.4409,0.8333,0.7967,0.8393,0.9736
5,Cassette,DART temporal,12,12,32921,0.7757,0.9095,0.4446,0.8158,0.7888,0.8550,0.9745
6,Cassette,CRNN pequeña,12,12,32921,0.7733,0.9162,0.4407,0.8545,0.7241,0.8751,0.9720
7,Cassette,DART + CRNN,12,12,32921,0.7949,0.9244,0.4596,0.8554,0.8073,0.8744,0.9778
8,Telemetry,DART,22,44,42691,0.7881,0.8174,0.5749,0.8400,0.7946,0.8860,0.8452
9,Telemetry,DART temporal,22,44,42691,0.7811,0.8070,0.5670,0.8277,0.7876,0.8898,0.8337


DART temporal frente a DART · diferencia -0.0043 · IC 95 % por sujeto [-0.0078, -0.0009]


DART + CRNN frente a DART · diferencia +0.0077 · IC 95 % por sujeto [+0.0037, +0.0121]
